In [1]:
# ═══════════════════════════════════════════════════════════════
# 2-MIN OPEN GATE — LIVE SCANNER
# ═══════════════════════════════════════════════════════════════
#
# Strategy:  2-MIN Open Gate (SMB Capital + Kevin's Playbook)
# Source:    "The Only Opening Momentum Strategy I'd Trade"
# Timeframe: 2-MIN candles
# Direction: Long only
#
# THE 3-SECOND GATE (check before market opens):
#   01. Prior Close
#   02. Yesterday's High
#   03. Pre-Market High
#   04. Opening Print (Strong 2-MIN Chart)
#
# GATE RULE — Opening Print must be:
#   ✅ Above Yesterday's High
#   ✅ Above Prior Close
#   ✅ BOTH or no trade — period
#
# THE TRIGGER:
#   2-MIN candle CLOSES above Pre-Market High
#   ON VOLUME (3+ RVOL)
#
# ENTRY:
#   Close of breakout candle
#   (actual market price — not pre-market snapshot)
#
# STOP:
#   Low of breakout candle
#
# FILTERS:
#   1. Upper wick < 50% of body
#   2. Green candle (close > open)
#   3. 3+ RVOL vs 5-day average same time
#   4. SPY above 10 AND 20 SMA (daily)
#   5. Room overhead — no resistance within 3%
#
# SKIP IT IF:
#   ❌ Upper wick too big
#   ❌ Candle closes red
#   ❌ Never closes above PM High
#
# EXIT FRAMEWORK:
#   Exit 1: 50% at 1R → stop moves to BE
#   Exit 2: Trail 50% on 9-EMA (2-MIN)
#            First 2-MIN close below 9-EMA → exit
#   EOD:    4PM hard exit — no overnight holds
#
# SIGNAL WINDOW:
#   9:30AM - 9:35AM ONLY
#   First candle closes at 9:32AM
#   No new entries after 9:35AM
#   Opening drive or nothing
#
# ATTEMPTS:
#   Max 1 per stock per day
#   First breakout is the only shot
#   After that — move on
#
# POSITION SIZING:
#   $100 risk per trade (1% of $10K)
#   Shares = $100 / stop distance
#
# BASKET:
#   Dynamic — updated daily from watchlist file
#   Historical performance (2024-2026):
#     AAPU  → 69.0% WR ← star performer
#     SNAP  → 62.5% WR ← consistent
#     TSLL  → 60.7% WR ← strong
#     AMZU  → 59.1% WR ← works here
#     SMCX  → your live example ✅
#     NVDL  → your live example ✅
#     VZ    → your live example ✅
#
# BACKTEST RESULTS (2024-01-01 to 2026-08-14):
#   Trades:      376
#   Win Rate:    52.4%
#   Total P&L:   +$4,012
#   Per Trade:   +$10.67
#   Max DD:      -$1,512
#   Return:      +40.1% on $10K
#
# LIVE EXAMPLES CONFIRMED:
#   SMCX  Aug 13 2026 → $13.00 → $15.71 ✅
#   NVDL  Aug 5/6 2026 → $34.83 → $35.54 ✅
#   VZ    Jul 28 2026 → $46.91 → $47.28 ✅
#   TSLZ  Jul 23 2026 → $15.35 → $16.34 ✅
#
# ═══════════════════════════════════════════════════════════════
#
# EVERY MORNING — DO THIS BEFORE RUNNING:
# ─────────────────────────────────────────
# 1. Update 2MIN_OG_watchlist.txt
#    → Add today's gap-up candidates
#    → Strong catalyst required
#    → Check pre-market volume
#
# 2. Open this notebook
#    Run All Cells (Ctrl+F9)
#
# 3. 9:15AM → Confirm Cell 3 shows:
#    ✅ Connected to Alpaca paper
#    ✅ Cash: $10,000.00
#
# 4. 9:25AM → Run Cell 12 → START
#    Scanner starts immediately
#    Waits for 9:30AM open
#
# 5. 9:30AM → Market opens
#    Scanner checks every 2 minutes
#    Gate check runs automatically
#    Signal fires if all conditions met
#
# 6. 9:35AM → Window closes
#    No new entries after this
#    Manage open positions until 4PM
#
# 7. 4:00PM → All positions closed
#    Run Cell 11 → daily_summary()
#
# ═══════════════════════════════════════════════════════════════

%pip install alpaca-py
%pip install pandas numpy python-dotenv


Note: you may need to restart the kernel to use updated packages.


In [2]:
# ─── Cell 2: Imports ─────────────────────────────────────────
import pandas as pd
import numpy as np
import os
import time as time_module
import datetime as dt
from datetime import datetime
from dotenv import load_dotenv

from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame, TimeFrameUnit
from alpaca.trading.client import TradingClient
from alpaca.trading.requests import MarketOrderRequest
from alpaca.trading.enums import OrderSide, TimeInForce

print("✅ Libraries imported")

✅ Libraries imported


In [3]:
# ─── Cell 3: Alpaca Connection ────────────────────────────────
dotenv_path = (
    r"c:\Users\kwgit\Desktop\Playbook"
    r"\Claude_2025_2026_Playbooks"
    r"\AlgoTrading\AlgoTrading\.env"
)
load_dotenv(dotenv_path, override=True)

API_KEY        = os.getenv("ALPACA_PAPER_API_KEY")
SECRET_KEY     = os.getenv("ALPACA_PAPER_SECRET_KEY")

data_client    = StockHistoricalDataClient(
    api_key    = API_KEY,
    secret_key = SECRET_KEY
)
trading_client = TradingClient(
    api_key    = API_KEY,
    secret_key = SECRET_KEY,
    paper      = True
)

account = trading_client.get_account()
print(f"✅ Alpaca paper connected")
print(f"   Cash:         ${float(account.cash):,.2f}")
print(f"   Buying power: ${float(account.buying_power):,.2f}")
print(f"   Paper:        confirmed ✅")

✅ Alpaca paper connected
   Cash:         $10,130.20
   Buying power: $40,520.80
   Paper:        confirmed ✅


In [4]:
# ─── Cell 4: Configuration ────────────────────────────────────

# ── Files ─────────────────────────────────────────────────────
BASE_PATH    = (
    r"c:\Users\kwgit\Desktop\Playbook"
    r"\Claude_2025_2026_Playbooks"
    r"\AlgoTrading\AlgoTrading"
)
WATCHLIST_FILE = BASE_PATH + r"\2MIN_OG_watchlist.txt"
CSV_FILE       = BASE_PATH + r"\2MIN_OG_alerts.csv"

# ── Session timing ────────────────────────────────────────────
PREMARKET_START  = dt.time(4, 0)
MARKET_OPEN      = dt.time(9, 30)
SIGNAL_WINDOW_END= dt.time(9, 35)  # no new entries after
EOD              = dt.time(16, 0)

# ── Strategy parameters ───────────────────────────────────────
DOLLAR_RISK      = 100      # $100 risk per trade
RVOL_THRESHOLD   = 3.0      # 3x relative volume
WICK_MAX_RATIO   = 0.50     # upper wick < 50% of body
RESISTANCE_PCT   = 0.03     # no resistance within 3%
EMA9_PERIOD      = 9        # 9-EMA for trail exit
SPY_SMA_FAST     = 10       # SPY 10 SMA
SPY_SMA_SLOW     = 20       # SPY 20 SMA
MAX_ATTEMPTS     = 1        # 1 attempt per stock per day

print(f"✅ Configuration loaded")
print(f"\n   Timing:")
print(f"   Pre-market:    {PREMARKET_START}")
print(f"   Market open:   {MARKET_OPEN}")
print(f"   Signal window: {MARKET_OPEN} - "
      f"{SIGNAL_WINDOW_END} ← entries only here")
print(f"   EOD:           {EOD}")
print(f"\n   Parameters:")
print(f"   Risk per trade: ${DOLLAR_RISK}")
print(f"   RVOL min:       {RVOL_THRESHOLD}x")
print(f"   Wick max:       {WICK_MAX_RATIO*100}% of body")
print(f"   Resistance:     {RESISTANCE_PCT*100}% clear overhead")
print(f"   Max attempts:   {MAX_ATTEMPTS} per stock per day")

✅ Configuration loaded

   Timing:
   Pre-market:    04:00:00
   Market open:   09:30:00
   Signal window: 09:30:00 - 09:35:00 ← entries only here
   EOD:           16:00:00

   Parameters:
   Risk per trade: $100
   RVOL min:       3.0x
   Wick max:       50.0% of body
   Resistance:     3.0% clear overhead
   Max attempts:   1 per stock per day


In [5]:
# ─── Cell 5: Watchlist Loader ─────────────────────────────────

def load_watchlist():
    """
    Load tickers from 2MIN_OG_watchlist.txt
    Update this file every morning with
    today's gap-up candidates

    Format: one ticker per line
    Lines starting with # are ignored
    """
    if not os.path.exists(WATCHLIST_FILE):
        with open(WATCHLIST_FILE, 'w') as f:
            f.write("# 2-MIN Open Gate Watchlist\n")
            f.write("# Update every morning\n")
            f.write("# One ticker per line\n")
            f.write("# Strong catalyst required\n")
            f.write("# Example:\n")
            f.write("# AAPU\n")
            f.write("# TSLL\n")
            f.write("# SNAP\n")
        print(f"✅ Created 2MIN_OG_watchlist.txt")
        print(f"   Add today's tickers before running")
        return []

    with open(WATCHLIST_FILE, 'r') as f:
        tickers = [
            line.strip().upper()
            for line in f
            if line.strip() and
            not line.strip().startswith('#')
        ]

    return tickers


# ── Setup CSV ─────────────────────────────────────────────────
csv_headers = [
    # Signal info
    'timestamp', 'ticker', 'signal_time',

    # Gate check
    'prev_day_high', 'prev_day_close',
    'open_price', 'gate_pass',

    # Pre-market
    'premarket_high', 'premarket_low',

    # Breakout candle
    'entry', 'stop', 'risk', 'shares',
    'risk_pct',

    # Filters
    'rvol', 'wick_ratio', 'upper_wick',
    'candle_body', 'is_green',
    'spy_above_sma', 'room_overhead',

    # Exit levels
    'target_1r',

    # You fill in after market
    'paper_traded', 'outcome',
    'exit_price', 'pnl',
    'exit_reason', 'notes'
]

if not os.path.exists(CSV_FILE):
    pd.DataFrame(columns=csv_headers).to_csv(
        CSV_FILE, index=False)
    print(f"✅ Created 2MIN_OG_alerts.csv")
else:
    existing = pd.read_csv(CSV_FILE)
    print(f"✅ CSV found — "
          f"{len(existing)} alerts logged so far")

print(f"\n✅ Watchlist loader ready")
watchlist = load_watchlist()
print(f"   Today's watchlist: {watchlist}")

✅ CSV found — 0 alerts logged so far

✅ Watchlist loader ready
   Today's watchlist: ['SNAP', 'MRVL', 'DKNG', 'PLTU', 'SMCX', 'NVDL', 'SNXX', 'AMZU', 'MSFU', 'AAPU', 'TSLL', 'SPCH', 'SSPC', 'MRNA', 'SKHX', 'MVLL', 'IBIT', 'CONL', 'AMDL', 'HOOG', 'SNDQ', 'CMG', 'CRMG', 'AFRM', 'GME']


In [6]:
# ─── Cell 6: Data Fetcher ─────────────────────────────────────

def fetch_bars(ticker, timeframe_minutes,
               lookback_days=5):
    """
    Fetch bars for one ticker
    Includes pre-market (4AM+) for complete session
    """
    try:
        start = datetime.now() - dt.timedelta(
            days=lookback_days)

        req = StockBarsRequest(
            symbol_or_symbols = [ticker],
            timeframe          = TimeFrame(
                timeframe_minutes,
                TimeFrameUnit.Minute
            ),
            start = start,
        )

        bars = data_client.get_stock_bars(req)
        df   = bars.df.reset_index()

        if len(df) == 0:
            return None

        df['timestamp'] = pd.to_datetime(df['timestamp'])
        if df['timestamp'].dt.tz is None:
            df['timestamp'] = df['timestamp'] \
                .dt.tz_localize('UTC')
        df['timestamp'] = df['timestamp'] \
            .dt.tz_convert('America/New_York')
        df = df.set_index('timestamp')
        df = df.sort_index()

        return df

    except Exception as e:
        print(f"   ❌ {ticker}: {str(e)[:50]}")
        return None


def fetch_daily_bars(ticker, lookback_days=30):
    """
    Fetch daily bars for SPY market filter
    and previous day high/close calculation
    """
    try:
        start = datetime.now() - dt.timedelta(
            days=lookback_days)

        req = StockBarsRequest(
            symbol_or_symbols = [ticker],
            timeframe          = TimeFrame(
                1, TimeFrameUnit.Day),
            start = start,
        )

        bars = data_client.get_stock_bars(req)
        df   = bars.df.reset_index()

        if len(df) == 0:
            return None

        df['timestamp'] = pd.to_datetime(df['timestamp'])
        if df['timestamp'].dt.tz is None:
            df['timestamp'] = df['timestamp'] \
                .dt.tz_localize('UTC')
        df['timestamp'] = df['timestamp'] \
            .dt.tz_convert('America/New_York')
        df = df.set_index('timestamp')
        df = df.sort_index()

        return df

    except Exception as e:
        print(f"   ❌ {ticker} daily: {str(e)[:50]}")
        return None


def get_premarket_data(df):
    """Pre-market bars (4AM-9:30AM) for today"""
    today    = datetime.now().date()
    pm_start = pd.Timestamp(
        year=today.year, month=today.month,
        day=today.day, hour=4, minute=0,
        tz='America/New_York'
    )
    pm_end = pd.Timestamp(
        year=today.year, month=today.month,
        day=today.day, hour=9, minute=30,
        tz='America/New_York'
    )
    return df[(df.index >= pm_start) &
              (df.index < pm_end)]


def get_market_data(df):
    """Market hours bars (9:30AM+) for today"""
    today     = datetime.now().date()
    mkt_start = pd.Timestamp(
        year=today.year, month=today.month,
        day=today.day, hour=9, minute=30,
        tz='America/New_York'
    )
    return df[df.index >= mkt_start]


print("✅ Data fetcher loaded")
print("   fetch_bars(ticker, timeframe_minutes)")
print("   fetch_daily_bars(ticker)")
print("   get_premarket_data(df)")
print("   get_market_data(df) — timezone fix applied")

✅ Data fetcher loaded
   fetch_bars(ticker, timeframe_minutes)
   fetch_daily_bars(ticker)
   get_premarket_data(df)
   get_market_data(df) — timezone fix applied


In [7]:
# ─── Cell 7: Indicator Calculator ────────────────────────────

# Fetch SPY daily once per session
spy_daily = None

def load_spy_filter():
    """
    Load SPY daily bars and calculate
    10 and 20 SMA for market filter
    Call once at startup
    """
    global spy_daily
    print("Loading SPY market filter...")
    df = fetch_daily_bars('SPY', lookback_days=30)

    if df is None:
        print("⚠️  SPY not loaded — filter disabled")
        spy_daily = None
        return

    df['sma10'] = df['close'].rolling(SPY_SMA_FAST).mean()
    df['sma20'] = df['close'].rolling(SPY_SMA_SLOW).mean()
    df['above_sma'] = (
        (df['close'] > df['sma10']) &
        (df['close'] > df['sma20'])
    )
    spy_daily = df
    latest    = df.iloc[-1]
    above     = latest['above_sma']
    print(f"✅ SPY loaded")
    print(f"   Close:  ${latest['close']:.2f}")
    print(f"   SMA10:  ${latest['sma10']:.2f}")
    print(f"   SMA20:  ${latest['sma20']:.2f}")
    print(f"   Filter: {'✅ PASS' if above else '❌ FAIL'}")


def get_spy_filter():
    """Check if SPY is above both SMAs today"""
    if spy_daily is None:
        return True  # default pass if not loaded
    latest = spy_daily.iloc[-1]
    return bool(latest['above_sma'])


def calculate_rvol(df_2min, signal_bar_time):
    """
    Calculate RVOL for the signal bar
    Compares bar volume to 5-day average
    at same time of day

    Returns rvol float or None
    """
    try:
        df = df_2min.copy()
        df['date']     = df.index.date
        df['time_str'] = df.index.strftime('%H:%M')

        # Get time of signal bar
        bar_time = signal_bar_time.strftime('%H:%M')

        # Get all bars at same time
        same_time = df[df['time_str'] == bar_time]

        if len(same_time) < 2:
            return None

        # Today's volume at this time
        today     = datetime.now().date()
        today_bar = same_time[same_time['date'] == today]
        if len(today_bar) == 0:
            return None

        today_vol = today_bar['volume'].iloc[0]

        # Prior 5 days average at same time
        prior = same_time[same_time['date'] < today]
        if len(prior) == 0:
            return None

        avg_vol = prior['volume'].tail(5).mean()
        if avg_vol == 0:
            return None

        return round(today_vol / avg_vol, 2)

    except Exception as e:
        return None


def calculate_9ema(df_2min):
    """Calculate 9-EMA on 2-MIN bars for trail exit"""
    df = df_2min.copy()
    df['ema9'] = df['close'].ewm(
        span=EMA9_PERIOD, adjust=False).mean()
    return df['ema9']


def get_previous_day_levels(ticker):
    """
    Get yesterday's high and close
    for the gate check
    Returns (prev_high, prev_close) or (None, None)
    """
    df = fetch_daily_bars(ticker, lookback_days=10)
    if df is None or len(df) < 2:
        return None, None

    # Second to last bar = yesterday
    yesterday  = df.iloc[-2]
    prev_high  = round(yesterday['high'], 4)
    prev_close = round(yesterday['close'], 4)

    return prev_high, prev_close


print("✅ Indicator calculator loaded")
print("   load_spy_filter()         → SPY market filter")
print("   get_spy_filter()          → True/False")
print("   calculate_rvol()          → RVOL float")
print("   calculate_9ema()          → 9-EMA series")
print("   get_previous_day_levels() → prev_high, prev_close")

✅ Indicator calculator loaded
   load_spy_filter()         → SPY market filter
   get_spy_filter()          → True/False
   calculate_rvol()          → RVOL float
   calculate_9ema()          → 9-EMA series
   get_previous_day_levels() → prev_high, prev_close


In [8]:
# ─── Cell 8: Signal Detector ──────────────────────────────────

# Track daily attempts per ticker
og_attempts = {}

def reset_og_attempts():
    global og_attempts
    og_attempts = {}
    print("✅ Open Gate attempt counters reset")


def check_og_signal(ticker):
    """
    Check one ticker for 2-MIN Open Gate signal

    Process:
      1. Gate check — open > prev_high AND prev_close
      2. SPY filter — above 10/20 SMA
      3. Fetch 2-MIN bars
      4. Check first market candle (9:30-9:32AM)
      5. Must close above PM High
      6. Apply all filters
      7. Return signal dict or None

    Returns signal dict if valid, None otherwise
    """
    # ── Attempt check ─────────────────────────────────────────
    attempts = og_attempts.get(ticker, 0)
    if attempts >= MAX_ATTEMPTS:
        return None

    # ── SPY market filter ─────────────────────────────────────
    if not get_spy_filter():
        return None

    # ── Fetch 2-MIN bars ──────────────────────────────────────
    df_2min = fetch_bars(ticker, 2)
    if df_2min is None or len(df_2min) < 50:
        return None

    # ── Get market and pre-market data ────────────────────────
    mkt_data = get_market_data(df_2min)
    pm_data  = get_premarket_data(df_2min)

    if len(mkt_data) == 0:
        return None

    if len(pm_data) == 0:
        return None

    # ── Pre-market high ───────────────────────────────────────
    premarket_high = pm_data['high'].max()
    premarket_low  = pm_data['low'].min()

    # ── Previous day levels ───────────────────────────────────
    prev_high, prev_close = get_previous_day_levels(ticker)
    if prev_high is None or prev_close is None:
        return None

    # ── Opening price ─────────────────────────────────────────
    open_price = mkt_data['open'].iloc[0]

    # ── Gate check ────────────────────────────────────────────
    # Opening print must be above BOTH:
    #   Yesterday's High
    #   Prior Close
    gate_pass = (open_price > prev_high and
                 open_price > prev_close)

    if not gate_pass:
        return None

    # ── Check latest market candle ────────────────────────────
    # Must be within signal window (9:30-9:35AM)
    latest_bar = mkt_data.iloc[-1]
    bar_time   = latest_bar.name.time()

    if bar_time >= SIGNAL_WINDOW_END:
        return None

    # ── Must close above pre-market high ─────────────────────
    close = latest_bar['close']
    if close <= premarket_high:
        return None

    # ── Filter 1 — Green candle ───────────────────────────────
    open_bar = latest_bar['open']
    if close <= open_bar:
        return None

    # ── Filter 2 — Wick filter ────────────────────────────────
    body       = abs(close - open_bar)
    upper_wick = latest_bar['high'] - max(close, open_bar)

    if body == 0:
        return None

    wick_ratio = upper_wick / body
    if wick_ratio > WICK_MAX_RATIO:
        return None

    # ── Filter 3 — RVOL ──────────────────────────────────────
    rvol = calculate_rvol(df_2min, latest_bar.name)
    if rvol is None or rvol < RVOL_THRESHOLD:
        return None

    # ── Filter 4 — Room overhead ──────────────────────────────
    # No resistance within 3% above entry
    # Use PM High as potential resistance
    # If PM High is within 3% above close → skip
    # (price already closed above it — not an issue)
    # Check if prev_day_high is within 3% above close
    overhead_clear = True
    if prev_high > close:
        distance = (prev_high - close) / close
        if distance < RESISTANCE_PCT:
            overhead_clear = False

    if not overhead_clear:
        return None

    # ── Valid signal ──────────────────────────────────────────
    entry    = close
    stop     = latest_bar['low']
    risk     = entry - stop

    if risk <= 0:
        return None
    if risk < entry * 0.001:
        return None
    if risk > entry * 0.05:
        return None

    shares   = max(round(DOLLAR_RISK / risk), 1)
    target_1r = round(entry + risk, 4)

    return {
        'timestamp':    datetime.now().strftime(
                        '%Y-%m-%d %H:%M:%S'),
        'ticker':       ticker,
        'signal_time':  str(latest_bar.name),

        # Gate
        'prev_day_high':  round(prev_high, 4),
        'prev_day_close': round(prev_close, 4),
        'open_price':     round(open_price, 4),
        'gate_pass':      gate_pass,

        # Pre-market
        'premarket_high': round(premarket_high, 4),
        'premarket_low':  round(premarket_low, 4),

        # Entry
        'entry':          round(entry, 4),
        'stop':           round(stop, 4),
        'risk':           round(risk, 4),
        'shares':         shares,
        'risk_pct':       round(risk/entry*100, 2),
        'target_1r':      target_1r,

        # Filters
        'rvol':           rvol,
        'wick_ratio':     round(wick_ratio, 2),
        'upper_wick':     round(upper_wick, 4),
        'candle_body':    round(body, 4),
        'is_green':       True,
        'spy_above_sma':  get_spy_filter(),
        'room_overhead':  overhead_clear,

        # Fill in after market
        'paper_traded':   False,
        'outcome':        '',
        'exit_price':     '',
        'pnl':            '',
        'exit_reason':    '',
        'notes':          ''
    }


def log_og_signal(signal):
    """Save signal to CSV"""
    df_new = pd.DataFrame([signal])
    if os.path.exists(CSV_FILE):
        df_existing = pd.read_csv(CSV_FILE)
        df_combined = pd.concat(
            [df_existing, df_new],
            ignore_index=True)
    else:
        df_combined = df_new

    df_combined.to_csv(CSV_FILE, index=False)


print("✅ Signal detector loaded")
print("   check_og_signal(ticker) → signal or None")
print()
print("   Signal requires ALL of:")
print("   ✅ Gate: open > prev_high AND prev_close")
print("   ✅ SPY above 10/20 SMA")
print("   ✅ 2-MIN close above PM High")
print("   ✅ Green candle")
print("   ✅ Wick ratio < 50% of body")
print("   ✅ 3+ RVOL")
print("   ✅ Room overhead (no resistance within 3%)")
print("   ✅ Within 9:30-9:35AM window")
print("   ✅ Risk within bounds (0.1% - 5%)")

✅ Signal detector loaded
   check_og_signal(ticker) → signal or None

   Signal requires ALL of:
   ✅ Gate: open > prev_high AND prev_close
   ✅ SPY above 10/20 SMA
   ✅ 2-MIN close above PM High
   ✅ Green candle
   ✅ Wick ratio < 50% of body
   ✅ 3+ RVOL
   ✅ Room overhead (no resistance within 3%)
   ✅ Within 9:30-9:35AM window
   ✅ Risk within bounds (0.1% - 5%)


In [9]:
# ─── Cell 9: Trade Executor + Position Manager ────────────────

og_positions = {}


def place_og_trade(signal):
    """
    Place paper trade for valid Open Gate signal

    Entry:  market order at signal close price
    Stop:   low of breakout candle
    Exit 1: 50% at 1R
    Exit 2: trail 50% on 9-EMA (2-MIN)
    """
    ticker  = signal['ticker']
    entry   = signal['entry']
    stop    = signal['stop']
    risk    = signal['risk']
    shares  = signal['shares']
    target  = signal['target_1r']

    # Already in position
    if ticker in og_positions:
        print(f"   ⚠️  {ticker} — already in position")
        return False

    try:
        # Place buy order
        order = trading_client.submit_order(
            MarketOrderRequest(
                symbol        = ticker,
                qty           = shares,
                side          = OrderSide.BUY,
                time_in_force = TimeInForce.DAY
            )
        )

        # Store position
        og_positions[ticker] = {
            'order_id':       order.id,
            'entry':          entry,
            'stop':           stop,
            'risk':           risk,
            'shares_total':   shares,
            'shares_remain':  shares,
            'target_1r':      target,
            'tp1_hit':        False,
            'stop_at_be':     False,
            'entry_time':     datetime.now(),
            'rvol':           signal['rvol'],
        }

        signal['paper_traded'] = True

        print(f"\n🟢 OPEN GATE PAPER TRADE — {ticker}")
        print(f"   Entry:     ${entry:.2f}")
        print(f"   Stop:      ${stop:.2f} "
              f"← breakout candle low")
        print(f"   Risk:      ${risk:.2f} × "
              f"{shares} = ${risk*shares:.2f}")
        print(f"   1R target: ${target:.2f}")
        print(f"   RVOL:      {signal['rvol']}x")
        print(f"   After 1R:  stop → BE ${entry:.2f}")
        print(f"   Trail:     50% on 9-EMA (2-MIN)")
        print(f"   Order ID:  {order.id}")
        return True

    except Exception as e:
        error = str(e)
        if 'buying_power' in error or \
                'insufficient' in error.lower():
            print(f"   💰 {ticker} — buying power exhausted")
        else:
            print(f"   ❌ Order failed {ticker}: {e}")
        return False


def manage_og_positions():
    """
    Monitor all open Open Gate positions

    Standard framework:
      → Check 1R target (50% exit)
      → After TP1: stop moves to BE
      → Trail remaining 50% on 9-EMA (2-MIN)
      → First 2-MIN close below 9-EMA → exit
      → Stop hit → exit all
      → EOD 4PM → exit all
    """
    if not og_positions:
        return

    for ticker, pos in list(og_positions.items()):
        try:
            # Fetch latest 2-MIN data
            df_2min = fetch_bars(ticker, 2)
            if df_2min is None:
                continue

            mkt_data = get_market_data(df_2min)
            if len(mkt_data) == 0:
                continue

            # Calculate 9-EMA
            ema9_series = calculate_9ema(df_2min)
            df_2min['ema9'] = ema9_series

            # Get latest market bars
            mkt_with_ema = df_2min[
                df_2min.index >= mkt_data.index[0]]

            price         = mkt_data['close'].iloc[-1]
            shares_remain = pos['shares_remain']
            entry         = pos['entry']
            risk          = pos['risk']
            target_1r     = pos['target_1r']
            tp1_hit       = pos['tp1_hit']

            if shares_remain <= 0:
                if ticker in og_positions:
                    del og_positions[ticker]
                continue

            # Current stop
            current_stop = entry if pos['stop_at_be'] \
                else pos['stop']

            # ── EOD exit ──────────────────────────────────────
            if datetime.now().time() >= EOD:
                trading_client.submit_order(
                    MarketOrderRequest(
                        symbol        = ticker,
                        qty           = shares_remain,
                        side          = OrderSide.SELL,
                        time_in_force = TimeInForce.DAY
                    )
                )
                pnl = (price - entry) * shares_remain
                print(f"\n🔔 EOD EXIT — {ticker} "
                      f"@ ${price:.2f} | "
                      f"P&L: ${pnl:.2f}")
                del og_positions[ticker]
                continue

            # ── Stop hit ──────────────────────────────────────
            if price <= current_stop:
                trading_client.submit_order(
                    MarketOrderRequest(
                        symbol        = ticker,
                        qty           = shares_remain,
                        side          = OrderSide.SELL,
                        time_in_force = TimeInForce.DAY
                    )
                )
                pnl = (current_stop - entry) * shares_remain
                stop_type = "BE" if pos['stop_at_be'] \
                    else "original"
                print(f"\n🔴 STOP ({stop_type}) — "
                      f"{ticker} @ ${current_stop:.2f} | "
                      f"P&L: ${pnl:.2f}")
                del og_positions[ticker]
                continue

            # ── TP1 — 50% at 1R ───────────────────────────────
            if not tp1_hit and price >= target_1r:
                lot_a = round(shares_remain * 0.50)
                lot_a = min(lot_a, shares_remain)

                if lot_a > 0:
                    trading_client.submit_order(
                        MarketOrderRequest(
                            symbol        = ticker,
                            qty           = lot_a,
                            side          = OrderSide.SELL,
                            time_in_force = TimeInForce.DAY
                        )
                    )
                    pnl = (target_1r - entry) * lot_a
                    print(f"\n✅ TP1 — {ticker} "
                          f"@ ${target_1r:.2f} | "
                          f"sold {lot_a} shares | "
                          f"+${pnl:.2f}")
                    pos['shares_remain'] -= lot_a
                    pos['tp1_hit']        = True
                    pos['stop_at_be']     = True
                    print(f"   Stop → BE: ${entry:.2f}")
                    print(f"   Trailing "
                          f"{pos['shares_remain']} "
                          f"shares on 9-EMA")

            # ── Trail on 9-EMA ────────────────────────────────
            # First 2-MIN close BELOW 9-EMA → exit
            elif tp1_hit and shares_remain > 0:
                if len(mkt_with_ema) >= 2:
                    last_close = mkt_with_ema[
                        'close'].iloc[-1]
                    last_ema9  = mkt_with_ema[
                        'ema9'].iloc[-1]

                    if last_close < last_ema9:
                        trading_client.submit_order(
                            MarketOrderRequest(
                                symbol        = ticker,
                                qty           = shares_remain,
                                side          = OrderSide.SELL,
                                time_in_force = TimeInForce.DAY
                            )
                        )
                        pnl = (price - entry) * shares_remain
                        print(f"\n✅ TRAIL EXIT "
                              f"(9-EMA) — {ticker} "
                              f"@ ${price:.2f} | "
                              f"P&L: ${pnl:.2f}")
                        del og_positions[ticker]
                        continue

            # ── Status ────────────────────────────────────────
            if shares_remain > 0 and \
                    ticker in og_positions:
                unrealized = (price-entry) * shares_remain
                ema9_val   = mkt_with_ema['ema9'].iloc[-1] \
                             if len(mkt_with_ema) > 0 \
                             else None
                stop_label = "BE" if pos['stop_at_be'] \
                    else "orig"
                ema_label  = f"${ema9_val:.2f}" \
                             if ema9_val else "building"
                print(f"   🟢 {ticker} ${price:.2f} | "
                      f"{shares_remain} shares | "
                      f"unrealized: ${unrealized:.2f} | "
                      f"stop ({stop_label}): "
                      f"${current_stop:.2f} | "
                      f"9-EMA: {ema_label}")

        except Exception as e:
            print(f"   ❌ Position error {ticker}: {e}")


print("✅ Trade executor loaded")
print("   place_og_trade(signal)  → places order")
print("   manage_og_positions()   → monitors positions")
print()
print("   Exit framework:")
print("   → 50% at 1R → stop to BE")
print("   → Trail 50% on 9-EMA (2-MIN)")
print("   → First 2-MIN close below 9-EMA → exit")
print("   → EOD 4PM hard exit")

✅ Trade executor loaded
   place_og_trade(signal)  → places order
   manage_og_positions()   → monitors positions

   Exit framework:
   → 50% at 1R → stop to BE
   → Trail 50% on 9-EMA (2-MIN)
   → First 2-MIN close below 9-EMA → exit
   → EOD 4PM hard exit


In [10]:
# ─── Cell 10: Main Scanner Loop ───────────────────────────────

def run_og_scanner(test_mode=False):
    """
    Main Open Gate scanner loop

    Session flow:
      4:00AM - 9:30AM → Pre-market monitoring
                         Load SPY filter
                         Monitor watchlist
                         Log pre-market activity
                         No trades placed

      9:30AM - 9:35AM → SIGNAL WINDOW
                         Check every 2 minutes
                         Gate check runs
                         All filters applied
                         Orders placed if valid
                         1 attempt per stock

      9:35AM - 4:00PM → Position management
                         Monitor open positions
                         50% at 1R → stop to BE
                         Trail on 9-EMA
                         EOD close at 4PM

    test_mode=True → 3 scans then stop
    """
    print(f"{'='*60}")
    print(f"  🚀 2-MIN OPEN GATE SCANNER")
    print(f"  {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"{'='*60}\n")

    reset_og_attempts()
    load_spy_filter()

    iteration = 0
    last_day  = datetime.now().date()

    while True:
        iteration    += 1
        now           = datetime.now()
        current_time  = now.time()

        # ── Reset at new trading day ──────────────────────────
        if now.date() != last_day:
            reset_og_attempts()
            load_spy_filter()
            last_day = now.date()
            print(f"\n🌅 New trading day — reset")

        # ── Load watchlist ────────────────────────────────────
        watchlist = load_watchlist()
        if not watchlist:
            print("⚠️  Watchlist empty — "
                  "add tickers to 2MIN_OG_watchlist.txt")
            time_module.sleep(60)
            continue

        print(f"\n{'─'*60}")
        print(f"  Scan #{iteration} — "
              f"{now.strftime('%H:%M:%S')}")
        print(f"  Watchlist: {watchlist}")
        print(f"{'─'*60}")

        # ── Session flags ─────────────────────────────────────
        in_premarket     = PREMARKET_START <= current_time \
                           < MARKET_OPEN
        in_signal_window = MARKET_OPEN <= current_time \
                           < SIGNAL_WINDOW_END
        past_window      = current_time >= SIGNAL_WINDOW_END
        past_eod         = current_time >= EOD

        # ── EOD ───────────────────────────────────────────────
        if past_eod:
            if og_positions:
                print("🔔 EOD — closing all positions")
                manage_og_positions()
            print("🔔 Market closed — stopping scanner")
            break

        # ── Past signal window — manage only ─────────────────
        if past_window:
            print(f"⏰ Signal window closed ({SIGNAL_WINDOW_END})")
            print(f"   No new entries — managing positions")
            if og_positions:
                print(f"   {len(og_positions)} "
                      f"open position(s):")
                manage_og_positions()
            else:
                print(f"   No open positions")
                if not test_mode:
                    print(f"   Run Cell 11 → daily_summary()")
                    break

        # ── Pre-market ────────────────────────────────────────
        elif in_premarket:
            print(f"🌙 Pre-market — monitoring")
            print(f"   SPY filter: "
                  f"{'✅ PASS' if get_spy_filter() else '❌ FAIL'}")

            # Check if gate conditions are buildable
            for ticker in watchlist:
                prev_high, prev_close = \
                    get_previous_day_levels(ticker)
                if prev_high and prev_close:
                    print(f"   {ticker:<8} "
                          f"prev_high: ${prev_high:.2f} | "
                          f"prev_close: ${prev_close:.2f}")

        # ── Signal window (9:30-9:35AM) ───────────────────────
        elif in_signal_window:
            print(f"🔔 SIGNAL WINDOW (9:30-9:35AM)")
            print(f"   Checking all tickers for "
                  f"breakout above PM High\n")

            for ticker in watchlist:
                attempts = og_attempts.get(ticker, 0)
                if attempts >= MAX_ATTEMPTS:
                    print(f"   {ticker:<8} → "
                          f"already attempted today")
                    continue

                print(f"   Checking {ticker}...")
                signal = check_og_signal(ticker)

                if signal:
                    # Log to CSV
                    log_og_signal(signal)

                    # Print signal details
                    print(f"\n   🎯 SIGNAL — {ticker}")
                    print(f"      Gate:     ✅ "
                          f"open ${signal['open_price']:.2f} > "
                          f"prev_high ${signal['prev_day_high']:.2f} "
                          f"AND prev_close "
                          f"${signal['prev_day_close']:.2f}")
                    print(f"      PM High:  "
                          f"${signal['premarket_high']:.2f}")
                    print(f"      Entry:    "
                          f"${signal['entry']:.2f}")
                    print(f"      Stop:     "
                          f"${signal['stop']:.2f}")
                    print(f"      Risk:     "
                          f"${signal['risk']:.2f} "
                          f"({signal['risk_pct']}%)")
                    print(f"      RVOL:     "
                          f"{signal['rvol']}x ✅")
                    print(f"      Wick:     "
                          f"{signal['wick_ratio']:.2f} ✅")
                    print(f"      1R:       "
                          f"${signal['target_1r']:.2f}")

                    # Place trade
                    traded = place_og_trade(signal)
                    if traded:
                        og_attempts[ticker] = \
                            og_attempts.get(ticker, 0) + 1
                        signal['paper_traded'] = True
                        print(f"   ✅ Order placed — {ticker}")
                    else:
                        print(f"   ⚠️  Order failed — "
                              f"attempt NOT counted")
                else:
                    print(f"   {ticker:<8} → no signal")

        # ── Too early ─────────────────────────────────────────
        else:
            print(f"⏳ Waiting for market open "
                  f"({MARKET_OPEN})")

        # ── Manage open positions every scan ──────────────────
        if og_positions:
            print(f"\n  📋 Managing "
                  f"{len(og_positions)} position(s):")
            manage_og_positions()

        # ── Test mode ─────────────────────────────────────────
        if test_mode and iteration >= 3:
            print(f"\n✅ Test complete — {iteration} scans")
            break

        # ── Sleep ─────────────────────────────────────────────
        if in_signal_window:
            print(f"\n  💤 Next scan in 2 minutes "
                  f"(signal window)...")
            time_module.sleep(120)
        else:
            print(f"\n  💤 Next scan in 5 minutes...")
            time_module.sleep(300)

    print(f"\n{'='*60}")
    print(f"  🏁 SCANNER STOPPED")
    print(f"  Run Cell 11 → daily_summary()")
    print(f"{'='*60}")


print("✅ Scanner loaded")
print("   run_og_scanner()               → live mode")
print("   run_og_scanner(test_mode=True) → 3 scans")
print()
print("   Signal window: 9:30AM - 9:35AM ONLY")
print("   Max attempts:  1 per stock per day")
print("   Scan interval: 2-MIN in window")
print("                  5-MIN all other times")

✅ Scanner loaded
   run_og_scanner()               → live mode
   run_og_scanner(test_mode=True) → 3 scans

   Signal window: 9:30AM - 9:35AM ONLY
   Max attempts:  1 per stock per day
   Scan interval: 2-MIN in window
                  5-MIN all other times


In [11]:
# ─── Cell 11: Daily Summary ───────────────────────────────────

def daily_summary():
    """
    Review today's Open Gate signals and trades
    Run after market close
    """
    if not os.path.exists(CSV_FILE):
        print("❌ No signals logged yet")
        return

    df    = pd.read_csv(CSV_FILE)
    today = datetime.now().strftime('%Y-%m-%d')
    df_today = df[df['timestamp'].str.startswith(today)]

    print(f"\n{'='*60}")
    print(f"  2-MIN OPEN GATE — DAILY SUMMARY")
    print(f"  {today}")
    print(f"{'='*60}")
    print(f"\n  Signals today: {len(df_today)}")

    if len(df_today) == 0:
        print("  No signals today")
        return

    # Signals fired
    print(f"\n  Signals:")
    for _, row in df_today.iterrows():
        traded = "✅ TRADED" if row['paper_traded'] \
                 else "📝 logged"
        print(f"  {row['ticker']:<8} "
              f"${row['entry']:.2f} | "
              f"stop ${row['stop']:.2f} | "
              f"1R ${row['target_1r']:.2f} | "
              f"RVOL {row['rvol']}x | "
              f"{traded}")

    # Results (if filled in)
    traded_today = df_today[df_today['paper_traded'] == True]
    if len(traded_today) > 0:
        print(f"\n  Traded: {len(traded_today)}")
        completed = traded_today[
            traded_today['outcome'] != '']
        if len(completed) > 0:
            total_pnl = pd.to_numeric(
                completed['pnl'],
                errors='coerce').sum()
            wins = completed[
                pd.to_numeric(completed['pnl'],
                errors='coerce') > 0]
            print(f"  Completed: {len(completed)}")
            print(f"  Winners:   {len(wins)}")
            print(f"  Total P&L: ${total_pnl:.2f}")

    print(f"\n  {'─'*40}")
    print(f"  Fill in outcome/exit_price/pnl/notes")
    print(f"  in 2MIN_OG_alerts.csv after market")
    print(f"{'='*60}")


def view_og_alerts(today_only=False,
                   min_rvol=0):
    """View all logged Open Gate alerts"""
    if not os.path.exists(CSV_FILE):
        print("❌ No alerts logged yet")
        return

    df = pd.read_csv(CSV_FILE)

    if today_only:
        today = datetime.now().strftime('%Y-%m-%d')
        df    = df[df['timestamp'].str.startswith(today)]

    if min_rvol > 0:
        df = df[pd.to_numeric(
            df['rvol'],
            errors='coerce') >= min_rvol]

    print(f"\n{'='*60}")
    print(f"  2-MIN OPEN GATE ALERT DATABASE")
    print(f"  Total alerts: {len(df)}")
    print(f"{'='*60}")

    if len(df) == 0:
        print("  No alerts match filters")
        return

    print(df[['timestamp', 'ticker', 'entry',
              'stop', 'risk', 'rvol',
              'target_1r', 'paper_traded',
              'outcome', 'pnl']].tail(
        20).to_string(index=False))

    print(f"\n  Filter options:")
    print(f"  view_og_alerts(today_only=True)")
    print(f"  view_og_alerts(min_rvol=5)")


print("✅ Summary tools loaded")
print("   daily_summary()     → today's recap")
print("   view_og_alerts()    → full database")

✅ Summary tools loaded
   daily_summary()     → today's recap
   view_og_alerts()    → full database


In [12]:
# ─── Cell 12: START ───────────────────────────────────────────
# Run ALL cells first (Ctrl+F9)
# Then run this cell to start the scanner
#
# Live mode — runs until 4PM:
run_og_scanner()
#
# Test mode — 3 scans only:
# run_og_scanner(test_mode=True)

  🚀 2-MIN OPEN GATE SCANNER
  2026-08-31 09:29:19

✅ Open Gate attempt counters reset
Loading SPY market filter...
✅ SPY loaded
   Close:  $769.35
   SMA10:  $767.34
   SMA20:  $769.22
   Filter: ✅ PASS

────────────────────────────────────────────────────────────
  Scan #1 — 09:29:20
  Watchlist: ['SNAP', 'MRVL', 'DKNG', 'PLTU', 'SMCX', 'NVDL', 'SNXX', 'AMZU', 'MSFU', 'AAPU', 'TSLL', 'SPCH', 'SSPC', 'MRNA', 'SKHX', 'MVLL', 'IBIT', 'CONL', 'AMDL', 'HOOG', 'SNDQ', 'CMG', 'CRMG', 'AFRM', 'GME']
────────────────────────────────────────────────────────────
🌙 Pre-market — monitoring
   SPY filter: ✅ PASS
   SNAP     prev_high: $5.57 | prev_close: $5.33
   MRVL     prev_high: $254.60 | prev_close: $241.45
   DKNG     prev_high: $24.94 | prev_close: $24.22
   PLTU     prev_high: $57.82 | prev_close: $57.27
   SMCX     prev_high: $13.45 | prev_close: $12.78
   NVDL     prev_high: $37.93 | prev_close: $37.07
   SNXX     prev_high: $14.16 | prev_close: $12.87
   AMZU     prev_high: $37.73 | prev